In [ ]:
"""
Concepts:

1. Bagging
2. Bootstrap Sampling
3. OOB Score
4. Feature Importance
5. Permutation Importance
6. Ensemble Learning
7. Variance Reduction
8. Calibration Analysis
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV,learning_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())
print(df["target"].value_counts(normalize=True))

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]

X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# Random Forest does NOT require scaling

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

rf = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)
rf.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================
val_pred = rf.predict(X_val)
val_prob = rf.predict_proba(X_val)[:,1]

print("\nBASELINE RANDOM FOREST")
accuracy_score(y_val,val_pred)
precision_score(y_val,val_pred)
recall_score(y_val,val_pred)
f1_score(y_val,val_pred)
roc_auc_score(y_val,val_prob)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(rf,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
print("\nCV Mean:",cv_scores.mean())
print("CV Std:",cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "n_estimators":[100,200,300,500],
    "max_depth":[3,5,10,15,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "max_features":["sqrt","log2"],
    "class_weight":[None,"balanced"]}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42,n_jobs=-1),param_distributions=param_grid,
                n_iter=25,cv=5,scoring="roc_auc",random_state=42,n_jobs=-1)

random_search.fit(X_train,y_train)
print(random_search.best_params_)
best_model = random_search.best_estimator_

In [ ]:
# =====================================================
# STEP 9 : VALIDATION check with best model
# =====================================================

val_pred = best_model.predict(X_val)
val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print("\nValidation ROC-AUC:",val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)
print("\nTrain ROC-AUC:",train_auc)
print("Validation ROC-AUC:",val_auc)

In [ ]:
# =====================================================
# STEP 11 : RANDOM FOREST ANALYSIS
# =====================================================

# ------------------------------------------
# OOB SCORE
# ------------------------------------------

oob_rf = RandomForestClassifier(n_estimators=300,oob_score=True,random_state=42,n_jobs=-1)
oob_rf.fit(X_train,y_train)
print("\nOOB Score:",oob_rf.oob_score_)

# ------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)
print(importance_df.head(15))
plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"][:15],importance_df["Importance"][:15])
plt.title("Feature Importance")
plt.show()

# ------------------------------------------
# PERMUTATION IMPORTANCE
# ------------------------------------------

perm = permutation_importance(best_model,X_val,y_val,random_state=42)

perm_df = pd.DataFrame({"Feature":X.columns,"Importance":perm.importances_mean}).sort_values(by="Importance",ascending=False).head(15)

# ------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------

cm = confusion_matrix(y_val,val_pred)
sns.heatmap(cm,annot=True,fmt="d")
plt.title("Confusion Matrix")
plt.show()

# ------------------------------------------
# CLASSIFICATION REPORT
# ------------------------------------------

print(classification_report(y_val,val_pred))

# ------------------------------------------
# ROC CURVE
# ------------------------------------------

fpr,tpr,_ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.title("ROC Curve")
plt.show()

# ------------------------------------------
# PRECISION RECALL CURVE
# ------------------------------------------

precision, recall, _ = precision_recall_curve(y_val,val_prob)
plt.plot(recall,precision)
plt.title("PR Curve")
plt.show()

# ------------------------------------------
# CALIBRATION CURVE
# ------------------------------------------

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.title("Calibration Curve")
plt.show()

# ------------------------------------------
# N_ESTIMATORS ANALYSIS
# ------------------------------------------

trees = [10,50,100,200,500]
scores = []

for n in trees:
    model = RandomForestClassifier(n_estimators=n,random_state=42,n_jobs=-1)
    model.fit(X_train,y_train)

    probs = model.predict_proba(X_val)[:,1]
    scores.append(roc_auc_score(y_val,probs))

plt.plot(trees,scores,marker='o')
plt.title("Trees vs ROC-AUC")
plt.show()

# ------------------------------------------
# MAX_DEPTH ANALYSIS
# ------------------------------------------

depths = [3,5,10,15,None]
scores = []

for depth in depths:
    model = RandomForestClassifier(max_depth=depth,random_state=42,n_jobs=-1)
    model.fit(X_train,y_train)

    probs = model.predict_proba(X_val)[:,1]
    scores.append(roc_auc_score(y_val,probs))

plt.plot(range(len(depths)),scores,marker='o')
plt.title("Depth Analysis")
plt.show()

# ------------------------------------------
# LEARNING CURVE
# ------------------------------------------

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)

plt.plot(train_sizes,np.mean(train_scores,axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores,axis=1),label="Validation")
plt.legend()
plt.title("Learning Curve")
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:,1]

print(classification_report(y_test,test_pred))

print("Test ROC-AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("\nDeployment Metrics")
print("CV ROC-AUC:",cv_scores.mean())
print("Validation ROC-AUC:",val_auc)
print("Test ROC-AUC:",roc_auc_score(y_test,test_prob))

# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

"""
1. What is Bagging?
2. What is Bootstrap Sampling?
3. Why Random Forest reduces variance?
4. Difference between Decision Tree and Random Forest?
5. What is OOB Score?
6. Why max_features is important?
7. Why does Random Forest not overfit easily?
8. Why is Random Forest better than a single tree?
9. Feature Importance vs Permutation Importance?
10. What happens if n_estimators increases?
11. What happens if max_depth increases?
12. Why scaling is unnecessary?
13. What is class_weight?
14. How does Random Forest handle missing values?
15. Why Random Forest can still overfit?
16. OOB Score vs Cross Validation?
17. Why is Random Forest considered robust?
18. Advantages of Random Forest?
19. Disadvantages of Random Forest?
20. Explain Random Forest from scratch.
"""